# Install Library

In [ ]:
!pip install mealpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.3/423.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.9/17.9 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 51.1 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.0 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.0 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.0 which is incompatible

In [ ]:
!pip install enoppy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 487.4 kB/s eta 0:00:00


# Import Library

In [ ]:
import os
import time
import json
import numpy as np
import pandas as pd
from pathlib import Path
from copy import deepcopy
from functools import partial
import concurrent.futures as parallel

from mealpy import FloatVar
from mealpy.utils.agent import Agent
from mealpy.optimizer import Optimizer
from mealpy.utils.problem import Problem
from mealpy.utils.termination import Termination
from mealpy.utils.validator import Validator

from typing import List, Tuple, Optional, Dict

# Connected to Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
path = "/content/drive/My Drive/Revision/"

import sys
sys.path.append(path)

# Engineering Problems

In [ ]:
from engineering_problems import engineering_problems

# Engineering Problem
names, functions, problems = engineering_problems()

print("Engineering problem count:", len(problems))
print("Problem names:", names)

Engineering problem count: 6
Problem names: ['Pressure Vessel', 'Welded Beam', 'Three Bar Truss', 'Multiple Disk', 'Tubular-Column', 'Corrugated-Bulkhead']


# Define Models

In [ ]:
from optimizers import NHO
from optimizers import ACO, DE, GA
from optimizers import HGSO, GWO, HHO, SSO
from optimizers import ACSA, BPBO, CHO, SRA
from optimizers import L_SHADE, CMA_ES, IMODE

try:
    from optimizers import LSHADEcnEpSin
    HAS_LSHADECNEPSIN = True
except Exception as e:
    HAS_LSHADECNEPSIN = False
    print("LSHADEcnEpSin import edilemedi, bu algoritma atlanacak:", e)

# Settings

In [ ]:
trial = 25

pop_size = 100
epoch = 500

n_jobs = None

SAVE_AS = "csv"
SAVE_CONVERGENCE = True
VERBOSE = True

save_root = path + "history/engineering"
os.makedirs(save_root, exist_ok=True)

runtime_csv_path = path + "history/engineering_runtime_all_algorithms_25trial.csv"

print("Save root:", save_root)

Save root: /content/drive/My Drive/Revision/history/engineering


# MultitaskEP

In [ ]:
class MultitaskEP:

    def __init__(self, algorithms=(), problems=(), terminations=None, modes=None, **kwargs):
        self.__set_keyword_arguments(kwargs)
        self.validator = Validator(log_to="console", log_file=None)
        self.algorithms = self.validator.check_list_tuple("algorithms", algorithms, "Optimizer")
        self.problems = self.validator.check_list_tuple("problems", problems, "Problem")
        self.terminations = terminations
        self.modes = modes
        self.n_algorithms = len(self.algorithms)
        self.m_problems = len(self.problems)

    def __set_keyword_arguments(self, kwargs):
        for key, value in kwargs.items():
            setattr(self, key, value)

    @staticmethod
    def export_to_dataframe(result: pd.DataFrame, save_path: str):
        result.to_pickle(f"{save_path}.pkl")

    @staticmethod
    def export_to_json(result: pd.DataFrame, save_path: str):
        result.to_json(f"{save_path}.json")

    @staticmethod
    def export_to_csv(result: pd.DataFrame, save_path: str):
        result.to_csv(f"{save_path}.csv", header=True, index=False)

    @staticmethod
    def _to_serializable_position(position):
        arr = np.asarray(position, dtype=float).reshape(-1)
        return json.dumps(arr.tolist())

    @staticmethod
    def _select_setting(setting, id_model, id_prob, default=None):
        if setting is None:
            return default
        if isinstance(setting, (str, Termination)):
            return setting
        if isinstance(setting, (list, tuple)):
            if len(setting) == 0:
                return default
            if len(setting) == 1:
                return setting[0]
            if len(setting) == id_model + 1:
                return setting[id_model]
            if len(setting) == id_prob + 1:
                return setting[id_prob]
        return setting

    def __run__(self, id_trial, model, problem, termination=None, mode="single"):
        result_from_solve = model.solve(problem, mode=mode, termination=termination)

        if isinstance(result_from_solve, Agent):
            best_position = result_from_solve.solution
            best_fitness = result_from_solve.target.fitness
        else:
            best_position, best_fitness = result_from_solve

        return {
            "id_trial": id_trial,
            "best_fitness": float(best_fitness),
            "best_position": best_position
        }

    def execute(self, n_trials=2, n_jobs=None, save_path="history", save_as="csv", save_convergence=False, verbose=False):

        n_trials = self.validator.check_int("n_trials", n_trials, [1, 100000])
        n_workers = None
        if (n_jobs is not None) and (n_jobs >= 1):
            n_workers = self.validator.check_int("n_jobs", n_jobs, [2, min(61, os.cpu_count() - 1)])

        save_as = self.validator.check_str("save_as", save_as, ["csv", "json", "dataframe"])
        export_function = getattr(self, f"export_to_{save_as}")

        for id_model, model in enumerate(self.algorithms):
            if not isinstance(model, Optimizer):
                print(f"Model: {id_model+1} is not an instance of Optimizer class.")
                continue

            path_best_fit = f"{save_path}/best_fit"
            path_best_position = f"{save_path}/best_position"
            Path(path_best_fit).mkdir(parents=True, exist_ok=True)
            Path(path_best_position).mkdir(parents=True, exist_ok=True)

            best_fit_model_results = {}
            best_position_model_results = {}

            for id_prob, problem in enumerate(self.problems):
                if not isinstance(problem, Problem):
                    if not type(problem) is dict:
                        print(f"Problem: {id_prob+1} is not an instance of Problem class or a Python dict.")
                        continue
                    else:
                        problem = Problem(**problem)

                termination = self._select_setting(self.terminations, id_model, id_prob, default=None)
                mode = self._select_setting(self.modes, id_model, id_prob, default="single")

                best_fit_trials = []
                best_position_trials = []
                trial_list = list(range(1, n_trials + 1))

                if n_workers is not None:
                    with parallel.ProcessPoolExecutor(n_workers) as executor:
                        list_results = executor.map(
                            partial(self.__run__, model=model, problem=problem, termination=termination, mode=mode),
                            trial_list
                        )
                        for result in list_results:
                            best_fit_trials.append(result["best_fitness"])
                            best_position_trials.append(self._to_serializable_position(result["best_position"]))
                            if verbose:
                                print(f"Solving problem: {problem.get_name()} using algorithm: {model.get_name()}, on the: {result['id_trial']} trial")
                else:
                    for idx in trial_list:
                        result = self.__run__(idx, model, problem, termination=termination, mode=mode)
                        best_fit_trials.append(result["best_fitness"])
                        best_position_trials.append(self._to_serializable_position(result["best_position"]))
                        if verbose:
                            print(f"Solving problem: {problem.get_name()} using algorithm: {model.get_name()}, on the: {result['id_trial']} trial")
                            print(f"Best Position: {result['best_position']}")
                            print(f"Best Fitness: {result['best_fitness']}")

                best_fit_model_results[problem.get_name()] = best_fit_trials
                best_position_model_results[problem.get_name()] = best_position_trials

            df_best = pd.DataFrame(best_fit_model_results)
            df_best_position = pd.DataFrame(best_position_model_results)

            export_function(df_best, f"{path_best_fit}/{model.get_name()}_best_fit")
            export_function(df_best_position, f"{path_best_position}/{model.get_name()}_best_position")

# Helper Functions

In [ ]:
def prepare_problems(problem_list):
    modified_problems = []

    for problem_dict in problem_list:
        if isinstance(problem_dict, Problem):
            modified_problems.append(problem_dict)
            continue

        new_problem_dict = problem_dict.copy()
        if "fit_func" in new_problem_dict:
            new_problem_dict["obj_func"] = new_problem_dict.pop("fit_func")
        modified_problems.append(new_problem_dict)

    return modified_problems


def build_algorithm_registry(epoch, pop_size):
    algorithms = {
        "NHO": NHO(epoch, pop_size),
        "ACO": ACO(epoch, pop_size),
        "DE": DE(epoch, pop_size),
        "GA": GA(epoch, pop_size),
        "GWO": GWO(epoch, pop_size),
        "HGSO": HGSO(epoch, pop_size),
        "HHO": HHO(epoch, pop_size),
        "SSO": SSO(epoch, pop_size),
        "ACSA": ACSA(epoch, pop_size),
        "BPBO": BPBO(epoch, pop_size),
        "CHO": CHO(epoch, pop_size),
        "SRA": SRA(epoch, pop_size),
        "L_SHADE": L_SHADE(epoch, pop_size),
        "CMA_ES": CMA_ES(epoch, pop_size),
        "IMODE": IMODE(epoch, pop_size),
    }

    if HAS_LSHADECNEPSIN:
        algorithms["LSHADEcnEpSin"] = LSHADEcnEpSin(epoch, pop_size)

    return algorithms


def run_single_algorithm_engineering(algo_name, algo, engineering_problem_list, save_path):
    os.makedirs(save_path, exist_ok=True)

    start_time = time.perf_counter()

    multitask = MultitaskEP(
        algorithms=[algo],
        problems=engineering_problem_list
    )

    multitask.execute(
        n_trials=trial,
        n_jobs=n_jobs,
        save_path=save_path,
        save_as=SAVE_AS,
        save_convergence=SAVE_CONVERGENCE,
        verbose=VERBOSE
    )

    runtime = time.perf_counter() - start_time

    return runtime

# Run Engineering Problems

In [ ]:
runtime_results = []

engineering_problem_list = prepare_problems(problems)
algorithms = build_algorithm_registry(epoch, pop_size)

print("=" * 80)
print(f"Engineering Problems | epoch={epoch} | pop_size={pop_size} | trial={trial}")
print("=" * 80)

for algo_name, algo in algorithms.items():
    print("\n" + "-" * 80)
    print(f"Running: {algo_name} | Engineering Problems | trial={trial}")
    print("-" * 80)

    algo_save_path = os.path.join(save_root, algo_name)

    try:
        runtime = run_single_algorithm_engineering(
            algo_name=algo_name,
            algo=algo,
            engineering_problem_list=engineering_problem_list,
            save_path=algo_save_path
        )

        runtime_results.append({
            "algorithm": algo_name,
            "suite": "engineering",
            "epoch": epoch,
            "pop_size": pop_size,
            "trial": trial,
            "n_jobs": n_jobs,
            "n_problems": len(engineering_problem_list),
            "runtime_seconds": runtime,
            "runtime_minutes": runtime / 60,
            "status": "OK",
            "error": ""
        })

        print(f"Runtime ({algo_name}, engineering): {runtime:.4f} seconds")

    except Exception as e:
        runtime_results.append({
            "algorithm": algo_name,
            "suite": "engineering",
            "epoch": epoch,
            "pop_size": pop_size,
            "trial": trial,
            "n_jobs": n_jobs,
            "n_problems": len(engineering_problem_list),
            "runtime_seconds": np.nan,
            "runtime_minutes": np.nan,
            "status": "ERROR",
            "error": str(e)
        })

        print(f"ERROR ({algo_name}, engineering): {e}")

runtime_df = pd.DataFrame(runtime_results)
runtime_df

Engineering Problems | epoch=500 | pop_size=100 | trial=25

--------------------------------------------------------------------------------
Running: NHO | Engineering Problems | trial=25
--------------------------------------------------------------------------------
Solving problem: Pressure Vessel using algorithm: NHO, on the: 1 trial
Best Position: [19.          9.         58.95849335 40.06825079]
Best Fitness: 7051.160785219929
Solving problem: Pressure Vessel using algorithm: NHO, on the: 2 trial
Best Position: [ 15.           8.          48.56969486 110.11622563]
Best Fitness: 6371.377297666613
Solving problem: Pressure Vessel using algorithm: NHO, on the: 3 trial
Best Position: [18.          9.         58.28959716 43.70192928]
Best Fitness: 6820.729516968009
Solving problem: Pressure Vessel using algorithm: NHO, on the: 4 trial
Best Position: [ 14.           7.          45.33573735 140.26507364]
Best Fitness: 6090.6603590220875
Solving problem: Pressure Vessel using algorithm: 

/usr/local/lib/python3.12/dist-packages/enoppy/paper_based/rwco_2020.py:1098: RuntimeWarning: divide by zero encountered in scalar divide
  gx[0] = x[1] / (np.sqrt(2) * x[0] ** 2 + 2 * x[0] * x[1]) * self.PP - self.xichma
/usr/local/lib/python3.12/dist-packages/enoppy/paper_based/rwco_2020.py:1099: RuntimeWarning: divide by zero encountered in scalar divide
  gx[0] = (np.sqrt(2) * x[0] + x[1]) / (np.sqrt(2) * x[0] ** 2 + 2 * x[0] * x[1]) * self.PP - self.xichma


Solving problem: Three Bar Truss using algorithm: NHO, on the: 1 trial
Best Position: [0.78833218 0.40921917]
Best Fitness: 263.8959299093645
Solving problem: Three Bar Truss using algorithm: NHO, on the: 2 trial
Best Position: [0.78865722 0.40829896]
Best Fitness: 263.89584381076406
Solving problem: Three Bar Truss using algorithm: NHO, on the: 3 trial
Best Position: [0.78879957 0.40789662]
Best Fitness: 263.89587206271614
Solving problem: Three Bar Truss using algorithm: NHO, on the: 4 trial
Best Position: [0.78864036 0.4083467 ]
Best Fitness: 263.8958473190062
Solving problem: Three Bar Truss using algorithm: NHO, on the: 5 trial
Best Position: [0.78938688 0.40623888]
Best Fitness: 263.8962147219306
Solving problem: Three Bar Truss using algorithm: NHO, on the: 6 trial
Best Position: [0.78883911 0.4077847 ]
Best Fitness: 263.89586312376736
Solving problem: Three Bar Truss using algorithm: NHO, on the: 7 trial
Best Position: [0.78806167 0.4099862 ]
Best Fitness: 263.89612051540206
So

/usr/local/lib/python3.12/dist-packages/enoppy/paper_based/rwco_2020.py:1098: RuntimeWarning: invalid value encountered in scalar divide
  gx[0] = x[1] / (np.sqrt(2) * x[0] ** 2 + 2 * x[0] * x[1]) * self.PP - self.xichma
/usr/local/lib/python3.12/dist-packages/enoppy/paper_based/rwco_2020.py:1099: RuntimeWarning: invalid value encountered in scalar divide
  gx[0] = (np.sqrt(2) * x[0] + x[1]) / (np.sqrt(2) * x[0] ** 2 + 2 * x[0] * x[1]) * self.PP - self.xichma
/usr/local/lib/python3.12/dist-packages/enoppy/paper_based/rwco_2020.py:1100: RuntimeWarning: divide by zero encountered in scalar divide
  gx[2] = 1 / (np.sqrt(2) * x[1] + x[0]) * self.PP - self.xichma


Solving problem: Three Bar Truss using algorithm: ACO, on the: 1 trial
Best Position: [0.78911107 0.40704095]
Best Fitness: 263.89841086955175
Solving problem: Three Bar Truss using algorithm: ACO, on the: 2 trial
Best Position: [0.78994273 0.40471149]
Best Fitness: 263.90069355711785
Solving problem: Three Bar Truss using algorithm: ACO, on the: 3 trial
Best Position: [0.78689861 0.41331202]
Best Fitness: 263.8997400440788
Solving problem: Three Bar Truss using algorithm: ACO, on the: 4 trial
Best Position: [0.79003824 0.40445543]
Best Fitness: 263.902101583153
Solving problem: Three Bar Truss using algorithm: ACO, on the: 5 trial
Best Position: [0.78892757 0.40756809]
Best Fitness: 263.8992232028482
Solving problem: Three Bar Truss using algorithm: ACO, on the: 6 trial
Best Position: [0.78839533 0.40907329]
Best Fitness: 263.89920237013575
Solving problem: Three Bar Truss using algorithm: ACO, on the: 7 trial
Best Position: [0.78874961 0.4080408 ]
Best Fitness: 263.89615775770227
Sol

/usr/local/lib/python3.12/dist-packages/enoppy/paper_based/pdo_2022.py:440: RuntimeWarning: divide by zero encountered in scalar divide
  f1 = 5.885*x[3]*(x[0] + x[2]) / (x[0] + np.sqrt(np.abs(x[2]**2 - x[1]**2)))


Solving problem: Corrugated-Bulkhead using algorithm: ACO, on the: 1 trial
Best Position: [92.28780791 31.50777476 92.3078286   1.5912448 ]
Best Fitness: 9.654418782714362
Solving problem: Corrugated-Bulkhead using algorithm: ACO, on the: 2 trial
Best Position: [79.2378559  32.56496146 79.25609432  1.39143662]
Best Fitness: 8.566928491672787


/usr/local/lib/python3.12/dist-packages/enoppy/paper_based/pdo_2022.py:440: RuntimeWarning: invalid value encountered in scalar divide
  f1 = 5.885*x[3]*(x[0] + x[2]) / (x[0] + np.sqrt(np.abs(x[2]**2 - x[1]**2)))


Görüntülenen çıkış son 5000 satıra kısaltıldı.
Solving problem: Pressure Vessel using algorithm: GWO, on the: 6 trial
Best Position: [ 13.           7.          42.09776564 176.72703254]
Best Fitness: 6061.714471951649
Solving problem: Pressure Vessel using algorithm: GWO, on the: 7 trial
Best Position: [ 13.           7.          42.07920063 176.94082788]
Best Fitness: 6063.592763205077
Solving problem: Pressure Vessel using algorithm: GWO, on the: 8 trial
Best Position: [ 13.           7.          42.06428154 177.075542  ]
Best Fitness: 6064.233054599826
Solving problem: Pressure Vessel using algorithm: GWO, on the: 9 trial
Best Position: [ 14.           7.          45.31416985 140.48675937]
Best Fitness: 6093.172498684126
Solving problem: Pressure Vessel using algorithm: GWO, on the: 10 trial
Best Position: [ 13.           7.          42.09133814 176.85932931]
Best Fitness: 6063.727436923681
Solving problem: Pressure Vessel using algorithm: GWO, on the: 11 trial
Best Position: [ 13.

,algorithm,suite,epoch,pop_size,trial,n_jobs,n_problems,runtime_seconds,runtime_minutes,status,error
0,NHO,engineering,500,100,25,None,6,2317.045762,38.617429,OK,
1,ACO,engineering,500,100,25,None,6,719.945751,11.999096,OK,
2,DE,engineering,500,100,25,None,6,915.905010,15.265084,OK,
3,GA,engineering,500,100,25,None,6,1340.184237,22.336404,OK,
4,GWO,engineering,500,100,25,None,6,864.087920,14.401465,OK,
5,HGSO,engineering,500,100,25,None,6,755.110455,12.585174,OK,
6,HHO,engineering,500,100,25,None,6,1159.803143,19.330052,OK,
7,SSO,engineering,500,100,25,None,6,568.714769,9.478579,OK,
8,ACSA,engineering,500,100,25,None,6,678.029384,11.300490,OK,
9,BPBO,engineering,500,100,25,None,6,571.017141,9.516952,OK,
